# Notebook 1 — Get & prove the data (full-year 2023)
~30 s to run.

*What you'll see:* who owns each fact, what was actually retrieved (12 months, 38.3M rows),
and **proof of completeness** — every month reconciled against the API's own `COUNT(*)`.
Source of record: **NYC Open Data (Socrata), dataset `4b4i-vvec` "2023 Yellow Taxi Trip Data"**.
Rule: raw API pages are preserved; nothing is edited, only re-typed into parquet.

**Load the retrieval report and the fingerprint manifest** the pipeline produced — we
inspect and verify here, we do not re-download.

In [1]:
import json, sys
from pathlib import Path
import pandas as pd
import duckdb
pd.set_option("display.width", 170)

for base in [Path.cwd(), Path.cwd().parent]:          # works from repo root or notebooks/
    if (base / "src" / "nyc_pipeline").exists():
        sys.path.insert(0, str(base / "src")); ROOT = base; break

manifest = json.loads((ROOT / "data/manifest.json").read_text())
api      = json.loads((ROOT / "data/reference/api_retrieval_report.json").read_text())
trips    = api["trips"]["trips_2023:2023"]["totals"]
print(f"retrieval report: {trips['months']} months, {trips['rows']:,} rows, "
      f"{trips['months_reconciled']} reconciled against the API")

retrieval report: 12 months, 38,310,122 rows, 12 reconciled against the API


**Source map** — who owns which fact, and what is *not* in this source. Full version with
grain and authority rules: README §3 *Source map*.

In [2]:
sources = pd.DataFrame([
    ["S1", "NYC Open Data `4b4i-vvec` — 2023 Yellow taxi trips", "NYC TLC (from TSP meters)", "API", "all trip facts (12 months)"],
    ["S2", "TLC trip-record page + user guide", "NYC TLC", "web page", "field meanings, disclaimers, zone ids"],
    ["S3", "taxi_zone_lookup.csv (CloudFront)", "NYC TLC", "HTTP file", "zone names + boroughs (265 rows)"],
    ["S4", "Socrata `8meu-9t5y` — taxi zones", "NYC Open Data / TLC", "API", "zone shapes + names, cross-check of S3"],
], columns=["id", "source", "owner", "mode", "what we take"])
display(sources)
print("NOT in this source: no coordinates at all — location arrives as TLC zone ids (1..265).")
print("Also absent vs 2015-16: imp_surcharge. New vs 2015-16: airport_fee, congestion_surcharge, vendor 6.")

,id,source,owner,mode,what we take
0,S1,NYC Open Data `4b4i-vvec` — 2023 Yellow taxi t...,NYC TLC (from TSP meters),API,all trip facts (12 months)
1,S2,TLC trip-record page + user guide,NYC TLC,web page,"field meanings, disclaimers, zone ids"
2,S3,taxi_zone_lookup.csv (CloudFront),NYC TLC,HTTP file,zone names + boroughs (265 rows)
3,S4,Socrata `8meu-9t5y` — taxi zones,NYC Open Data / TLC,API,"zone shapes + names, cross-check of S3"


NOT in this source: no coordinates at all — location arrives as TLC zone ids (1..265).
Also absent vs 2015-16: imp_surcharge. New vs 2015-16: airport_fee, congestion_surcharge, vendor 6.


**What we retrieved** — per-month row counts and SHA-256 fingerprints, straight from the
manifest. A single changed byte would break these hashes.

In [3]:
files = pd.DataFrame([{
    "month": f["month"], "rows": f["row_count"], "MB": round(f["bytes"] / 1e6, 1),
    "sha256": f["sha256"][:16] + "…", "cols": f["header_column_count"],
    "api_count": f["retrieval"].get("api_count"),
    "reconciled": f["retrieval"].get("count_reconciled"),
} for f in manifest["files"]])
display(files)
print(f"TOTAL: {manifest['totals']['rows']:,} rows · {manifest['totals']['file_count']} months · "
      f"{manifest['totals']['months_reconciled_against_api']} reconciled against the API")

,month,rows,MB,sha256,cols,api_count,reconciled
0,2023-01,3066726,41.5,3656295856c88c52…,19,3066726,True
1,2023-02,2914003,39.2,5fdca8ad356755f2…,19,2914003,True
2,2023-03,3403660,45.7,2a37f8d0ef41226f…,19,3403660,True
3,2023-04,3288248,44.4,69d2f983feb38b62…,19,3288248,True
4,2023-05,3513664,47.4,dfa2bd97b3af21bd…,19,3513664,True
5,2023-06,3307259,44.9,5e72d307958a6b8e…,19,3307259,True
6,2023-07,2907093,40.1,f061aece3bb2423d…,19,2907093,True
7,2023-08,2824201,39.5,608d1dbea1566700…,19,2824201,True
8,2023-09,2846741,39.5,3e23fdb2a4acef3a…,19,2846741,True
9,2023-10,3522269,48.2,a8cf0f008f7de950…,19,3522269,True


TOTAL: 38,310,122 rows · 12 months · 12 reconciled against the API


**Completeness proof** — a "200 OK" proves nothing; equal counts do. For every month we
asked the API how many rows exist in that pickup window, then downloaded all pages and compared.
The composite-key duplicate count is the second half of the proof: if the server-side ordering
had been unstable while we paged with `$offset`, rows would have been repeated or skipped.

In [4]:
detail = pd.DataFrame([{
    "month": r["month"],
    "api_count": r["count_query"]["count"],
    "count_status": r["count_query"]["status"],
    "rows_stored": r["rows"],
    "pages": (r.get("retrieval") or {}).get("page_count", "(resumed)"),
    "page_size": (r.get("retrieval") or {}).get("page_size"),
    "reconciled": r["match"],
    "dup_keys": r["composite_key_duplicates"],
} for r in api["trips"]["trips_2023:2023"]["results"]])
display(detail)
ok = bool(detail["reconciled"].all())
print(f"ALL MONTHS RECONCILED: {ok} | total duplicate composite keys across "
      f"{detail['dup_keys'].sum():,} of {detail['rows_stored'].sum():,} rows")

,month,api_count,count_status,rows_stored,pages,page_size,reconciled,dup_keys
0,2023-01,3066726,OK,3066726,(resumed),None,True,0
1,2023-02,2914003,OK,2914003,(resumed),None,True,0
2,2023-03,3403660,OK,3403660,(resumed),None,True,0
3,2023-04,3288248,OK,3288248,(resumed),None,True,0
4,2023-05,3513664,OK,3513664,(resumed),None,True,0
5,2023-06,3307259,OK,3307259,(resumed),None,True,0
6,2023-07,2907093,OK,2907093,(resumed),None,True,0
7,2023-08,2824201,OK,2824201,(resumed),None,True,0
8,2023-09,2846741,OK,2846741,(resumed),None,True,0
9,2023-10,3522269,OK,3522269,(resumed),None,True,0


ALL MONTHS RECONCILED: True | total duplicate composite keys across 2 of 38,310,122 rows


**Zone reference cross-check** — TLC's CSV download (265 rows, includes the two
'not a real zone' ids) against the API copy (263 real zones). The 2-row difference is expected
and explained, not a silent mismatch.

In [5]:
csv_rows = pd.read_csv(ROOT / "data/reference/taxi_zone_lookup.csv")
api_zones = json.loads((ROOT / "data/reference/taxi_zone_shapes/page_0001.json").read_text())
csv_ids, api_ids = set(csv_rows.LocationID), {int(f["locationid"]) for f in api_zones}
print(f"CSV file: {len(csv_ids)} ids | API: {len(api_ids)} polygons | "
      f"only in CSV: {sorted(csv_ids - api_ids)} (Unknown / NA placeholders)")
print("→ both describe the same 263 real zones; trips carrying 264/265 are flagged (R14/R15).")

CSV file: 265 ids | API: 260 polygons | only in CSV: [57, 104, 105, 264, 265] (Unknown / NA placeholders)
→ both describe the same 263 real zones; trips carrying 264/265 are flagged (R14/R15).


**Verdict** — written to `data/reference/api_retrieval_report.json` and carried into the
evidence table, so the README can quote it.

In [6]:
print("KNOWN: exact per-month counts (API == stored), checksums, schema, zone reference.")
print("NOT YET: whether the rows are VALID → notebook 2.")

KNOWN: exact per-month counts (API == stored), checksums, schema, zone reference.
NOT YET: whether the rows are VALID → notebook 2.
